In [1]:
import sys, os
import numpy as np
#sys.path.append(os.path.join(os.path.split(os.path.abspath(__file__))[0],'../read_parameters'))
sys.path.append('../read_parameters')
import read_parameters

In [2]:
def wall_potential(wall_type, general_coordinate,\
                    parameter_file='MoREST.in'):
    '''
    wall_potential will read the positions of atoms and then add forces of the potential on the atoms.
    INPUT:
    wall_type:                The type of wall potential, e.g. plane, sphere, point, line
    general_coordinate:       The positions vector of atoms in the system.
    parameter_file:           The path and name of the parameter file.
    OUTPUT:
    wall_force: the forces of the wall potential on the atoms
    '''
    
    log_morest = open('MoREST.log','a')
    wall_parameters = np.load('MoREST_wall_parameters.npy',allow_pickle=True).item()
    
    if not wall_parameters['Collective_variable']:
        if wall_parameters['Wall_type'] in ['Plane_opaque_wall', 'plane_opaque_wall']:
            log_morest.write('The defination of the plane opaque wall: Point in plane, Normal vector\n')
            log_morest.write(self.wall_parameters['Plane_wall_point'],self.wall_parameters['Plane_wall_normal_vector'])
            log_morest.write('\n')
            log_morest.write('The plane opaque wall potential and force on atoms: XYZ coordinate, Potential, Forces\n')
            wall_force = []
            for i_coordinate in general_coordinate:
                i_wall_force, i_wall_potential = Plane_wall.plane_opaque_wall().get_opaque_wall_force_potential(i_coordinate)
                wall_force.append(i_wall_force)
                log_morest.write(i_coordinate, i_wall_potential, i_wall_force)
                log_morest.write('\n')
            return np.array(wall_force)
        
        if wall_parameters['Wall_type'] in ['Plane_translucent_wall', 'plane_translucent_wall']:
            log_morest.write('The defination of the plane translucent wall: Point in plane, Normal vector\n')
            log_morest.write(self.wall_parameters['Plane_wall_point'],self.wall_parameters['Plane_wall_normal_vector'])
            log_morest.write('\n')
            log_morest.write('The plane translucent wall potential and force on atoms: XYZ coordinate, Potential, Forces\n')
            wall_force = []
            for i_coordinate in general_coordinate:
                i_wall_force, i_wall_potential = Plane_wall.plane_translucent_wall().get_translucent_wall_force_potential(i_coordinate)
                wall_force.append(i_wall_force)
                log_morest.write(i_coordinate, i_wall_potential, i_wall_force)
                log_morest.write('\n')
            return np.array(wall_force)
    
        else:
            log_morest.write('No wall type was matched.\n')
            log_morest.close()
            return np.array([0])
        
    else:
        general_coordinate = CV_to_xyz(general_coordinate) # TODO conversion function is not exist.

## Plane wall
### Opaque plane wall
#### The potential of a opaque plane wall:
$$ U_{opaque} = \left\{\begin{matrix}
\frac{a}{c^2} \cdot \frac{1}{MAX(\left \| \vec{gc}-\vec{b} \right \|,10^{-6})} \cdot (\left \| \vec{gc}-\vec{b} \right \| -c)^2 , & \left \| \vec{gc}-\vec{b} \right \| \leqslant c \\ 
0 , & \left \| \vec{gc}-\vec{b} \right \| \geqslant c
\end{matrix}\right. $$
where $MAX$ function return a maximum number between $|gc-b|$ and $10^{-6}$. Parameter $a$ determines the increasing scaling of the potential when the position is closing to the wall, while $b$ indicates the position of the wall, and $c$ is the length between the wall and the position of the potential decays to $0$.

#### The force of a opaque plane wall:
$$\vec{F}_{opaque} = - \frac{\partial U_{opaque}}{\partial \left \| \vec{gc}-\vec{b} \right \|} \cdot \frac{\vec{gc}-\vec{b}}{\left \| \vec{gc}-\vec{b} \right \|}
= \left\{\begin{matrix}
\frac{a}{c^2} \cdot (\frac{c^2}{\left \| \vec{gc}-\vec{b} \right \|^2} -1) \cdot \frac{\vec{gc}-\vec{b}}{\left \| \vec{gc}-\vec{b} \right \|} , & \left \| \vec{gc}-\vec{b} \right \| \leqslant c \\ 
\vec{0} , & \left \| \vec{gc}-\vec{b} \right \| \geqslant c
\end{matrix}\right. $$

### Translucent plane wall
#### The potential of a translucent plane wall:
$$U_{translucent} = \left\{\begin{matrix}
\frac{a}{c^4} \cdot (2c \cdot \left \| \vec{gc}-\vec{b} \right \| +c^2) \cdot (\left \| \vec{gc}-\vec{b} \right \| -c)^2 , & \left \| \vec{gc}-\vec{b} \right \| \leqslant c \\ 
0 , & \left \| \vec{gc}-\vec{b} \right \| \geqslant c
\end{matrix}\right.$$
where $a$ is the potential value on the postion of the wall, while $b$ is the position of the wall, and $c$ is the length between the wall and the position of the potential decays to $0$.

#### The force of a translucent plane wall:
$$\vec{F}_{translucent} = - \frac{\partial U_{translucent}}{\partial \left \| \vec{gc}-\vec{b} \right \|} \cdot \frac{\vec{gc}-\vec{b}}{\left \| \vec{gc}-\vec{b} \right \|}
= \left\{\begin{matrix}
6 \cdot \frac{a}{c^4} \cdot (c^2 \left \| \vec{gc}-\vec{b} \right \| - c \left \| \vec{gc}-\vec{b} \right \|^2) \cdot \frac{\vec{gc}-\vec{b}}{\left \| \vec{gc}-\vec{b} \right \|} , & \left \| \vec{gc}-\vec{b} \right \| \leqslant c \\ 
\vec{0} , & \left \| \vec{gc}-\vec{b} \right \| \geqslant c
\end{matrix}\right.$$

### General coordinate ($\vec{gc}$) and the position of the wall ($\vec{b}$)
$\vec{gc}$ and $\vec{b}$ are vectors in describing the positions of atoms and the position of the wall respectively. Both $\vec{gc}$ and $\vec{b}$ can be either Cartesian coordinates or any collective variables. In calculation, $\vec{gc}$ and $\vec{b}$ should be the same type of coordinate(Cartesian or CV).

### On the defination of the plane with point ($\vec{p}$) and normal vector ($\vec{n}$) in Cartesian coordinate
$$\vec{gc}-\vec{b} = (\vec{r} - \vec{p}) \cdot \frac{\vec{n}}{\left \| \vec{n} \right \|}, $$
where $\vec{r}$ is the position on which will be calculated the force of in the Cartesian coordinate. Vector $\vec{p}$ is the Cartesian coordinate of the point in the plane, while vector $\vec{n}$ is the normal vector of the plane. Normal vector $\vec{n}$ should be normalized in advance.

In [1]:
import numpy as np

class plane_opaque_wall:
    '''
    Adding a plane opaque wall with the potential in the system. The plane is defined by a point in the plane and
    a normal vector of the plane.
    INPUT:
    xyz_coordinate:        The xyz_coordinate is get or conversed from the general_coordinate which can be either cartesian coordinates or 
                               any collective variables. 
    '''
        
    def __init__(self):
        self.wall_parameters = np.load('MoREST_wall_parameters.npy',allow_pickle=True).item()
        
    def get_opaque_wall_force_potential(self, xyz_coordinate):
        # calculate the force and potential on an atom with xyz_coordinate
        a = self.wall_parameters['Plane_wall_scaling']
        c = self.wall_parameters['Plane_wall_scope']
        vec_gc_b = np.dot((xyz_coordinate - self.wall_parameters['Plane_wall_point']),self.wall_parameters['Plane_wall_normal_vector'])
        norm_gc_b = np.linalg.norm(vec_gc_b)
        if norm_gc_b > c:
            return np.zeros(np.shape(xyz_coordinate)), 0.
        else:
            prefactor = a/(c^2)
            wall_force = prefactor * ((c^2)/(norm_gc_b^2)-1) * vec_gc_b/norm_gc_b
            wall_potential = prefactor * (1/norm_gc_b) * (norm_gc_b - c)^2
            return wall_force, wall_potential
    
    
class plane_translucent_wall:
    '''
    Adding a plane translucent wall with the potential in the system. The plane is defined by a point on the plane and
    a normal vector of the plane.
    INPUT:
    xyz_coordinate:        The xyz_coordinate is get or conversed from the general_coordinate which can be either cartesian coordinates or 
                               any collective variables. 
    '''
        
    def __init__(self):
        self.wall_parameters = np.load('MoREST_wall_parameters.npy',allow_pickle=True).item()
        
    def get_translucent_wall_force_potential(self, xyz_coordinate):
        # calculate the force and potential on an atom with xyz_coordinate
        a = self.wall_parameters['Plane_wall_scaling']
        c = self.wall_parameters['Plane_wall_scope']
        vec_gc_b = np.dot((xyz_coordinate - self.wall_parameters['Plane_wall_point']),self.wall_parameters['Plane_wall_normal_vector'])
        norm_gc_b = np.linalg.norm(vec_gc_b)
        if norm_gc_b > c:
            return np.zeros(np.shape(xyz_coordinate)), 0.
        else:
            prefactor = a/(c^4)
            wall_force = 6 * prefactor * (c^2 * norm_gc_b - c * norm_gc_b^2) * vec_gc_b/norm_gc_b
            wall_potential = prefactor * (2*c * norm_gc_b + c^2) * (norm_gc_b -c )^2
            return wall_force, wall_potential